# 03 IFC + BCF Viewer

Interaktive PoC-Oberfläche mit `ipywidgets` zum Laden von IFC/BCF, Topic-Auswahl und GUID-Highlighting.

In [ ]:
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

from openbim_viewer.viewer_adapter import IFCViewerAdapter
from openbim_viewer.ifc_loader import load_ifc, build_ifc_index
from openbim_viewer.bcf_loader import extract_bcf_topics, extract_bcf_viewpoints
from openbim_viewer.mapping import map_bcf_to_ifc


In [ ]:
# --- UI Elemente ---
ifc_path_input = widgets.Text(description='IFC_PATH', placeholder='z. B. data/small.ifc', layout=widgets.Layout(width='700px'))
bcf_path_input = widgets.Text(description='BCF_PATH', placeholder='z. B. data/issues.bcfzip', layout=widgets.Layout(width='700px'))

load_ifc_btn = widgets.Button(description='Load IFC', button_style='primary')
load_bcf_btn = widgets.Button(description='Load BCF', button_style='info')

topic_dropdown = widgets.Dropdown(description='BCF Topic', options=[], layout=widgets.Layout(width='700px'))
topic_text = widgets.Textarea(description='Topic Info', value='', layout=widgets.Layout(width='700px', height='220px'))

viewer_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd'))
log_out = widgets.Output(layout=widgets.Layout(border='1px solid #eee'))

# --- State ---
adapter = IFCViewerAdapter()
viewer = None
ifc_file = None
ifc_index = {}
topics = []
viewpoints = []
mapping_df = None

topic_by_guid = {}
topic_to_guids = {}


In [ ]:
def _build_topic_info(topic: dict, guids: list[str]) -> str:
    title = topic.get('title') or '(ohne Titel)'
    status = topic.get('status') or '(ohne Status)'
    comments = topic.get('comments') or []
    comments_text = '\n'.join(f'- {c}' for c in comments) if comments else '(keine Kommentare)'
    guid_text = '\n'.join(f'- {g}' for g in guids) if guids else '(keine GUIDs)'
    return (
        f'Title: {title}\n'
        f'Status: {status}\n\n'
        f'Kommentare:\n{comments_text}\n\n'
        f'Referenzierte GUIDs:\n{guid_text}'
    )

def _on_load_ifc(_):
    global viewer, ifc_file, ifc_index
    with log_out:
        log_out.clear_output()
        try:
            path = Path(ifc_path_input.value)
            ifc_file = load_ifc(path)
            ifc_index = build_ifc_index(ifc_file)
            viewer = adapter.show(path)
            print(f'IFC geladen: {path}')
            print(f'IfcProduct im Index: {len(ifc_index)}')
            with viewer_out:
                viewer_out.clear_output()
                display(viewer)
        except Exception as exc:
            print(f'Fehler beim IFC-Laden: {exc}')

def _on_load_bcf(_):
    global topics, viewpoints, mapping_df, topic_by_guid, topic_to_guids
    with log_out:
        log_out.clear_output()
        try:
            path = Path(bcf_path_input.value)
            topics = extract_bcf_topics(path)
            viewpoints = extract_bcf_viewpoints(path)
            mapping_df = map_bcf_to_ifc(topics, viewpoints, ifc_index)

            topic_by_guid = {t.get('topic_guid'): t for t in topics if t.get('topic_guid')}
            topic_to_guids = {}
            if mapping_df is not None and not mapping_df.empty:
                for topic_guid, df_group in mapping_df.groupby('topic_guid', dropna=True):
                    guids = [g for g in df_group['referenced_ifc_guid'].dropna().unique().tolist() if g]
                    topic_to_guids[topic_guid] = guids

            options = []
            for t in topics:
                t_guid = t.get('topic_guid')
                if not t_guid:
                    continue
                t_title = t.get('title') or '(ohne Titel)'
                options.append((f'{t_title} [{t_guid}]', t_guid))

            topic_dropdown.options = options
            print(f'BCF geladen: {path}')
            print(f'Topics: {len(topics)}, Viewpoints: {len(viewpoints)}')
            print(f'Topics im Dropdown: {len(options)}')
        except Exception as exc:
            print(f'Fehler beim BCF-Laden: {exc}')

def _on_topic_change(change):
    if change.get('name') != 'value' or change.get('new') is None:
        return

    topic_guid = change['new']
    guids = topic_to_guids.get(topic_guid, [])
    topic = topic_by_guid.get(topic_guid, {})

    # geforderter Aufruf
    if viewer is not None and hasattr(viewer, 'highlight_guids'):
        try:
            viewer.highlight_guids(guids)
        except Exception:
            pass

    topic_text.value = _build_topic_info(topic, guids)

load_ifc_btn.on_click(_on_load_ifc)
load_bcf_btn.on_click(_on_load_bcf)
topic_dropdown.observe(_on_topic_change, names='value')


In [ ]:
ui = widgets.VBox([
    ifc_path_input,
    widgets.HBox([load_ifc_btn, load_bcf_btn]),
    bcf_path_input,
    topic_dropdown,
    topic_text,
    widgets.HTML('<b>Viewer</b>'),
    viewer_out,
    widgets.HTML('<b>Log</b>'),
    log_out,
] )
display(ui)